In [ ]:
"""
==============================================================================
CPT Calculation — ACTUAL DAG from Causal Discovery
==============================================================================
"""

import pandas as pd
import numpy as np
from itertools import product
from scipy.stats import beta as beta_dist
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 0. LOAD DATA
# ============================================================================
df = pd.read_csv(
    r"C:\Users\55279\Desktop\Mestrado\0.Thesis\Codes\Approach_3\NTSB\accident_topic_matrix_labeled.csv")
N = len(df)
print(f"Loaded: {N} accidents × {df.shape[1]} topics\n")

# ============================================================================
# 1. TOPIC LABELS 
# ============================================================================
topic_labels = {
    'topic_0':  'Passenger & Crew Injuries',
    'topic_1':  'Aircraft Structural Damage',
    'topic_2':  'ATC Communication Issues',
    'topic_4':  'Landing Gear Malfunctions',
    'topic_6':  'Fatigue Crack',
    'topic_7':  'Turbulence Encounter',
    'topic_8':  'Fuel System Installation Issues',
    'topic_9':  'Runway & Taxiway Incidents',
    'topic_10': 'Electrical Wiring Damage',
    'topic_11': 'Fan Blade Failure',
    'topic_13': 'Aircraft Electrical System Faults',
    'topic_14': 'High EGT',
    'topic_15': 'Turbine & Compressor Damage',
    'topic_16': 'Smoke or Odor in Cabin',
    'topic_18': 'Hydraulic System Pressure Issues',
}

def label(t):
    return f"{t} ({topic_labels.get(t, '?')})"

# ============================================================================
# 2. DAG EDGES 
# ============================================================================

dag_edges = [
    ('topic_9', 'topic_0'),    # Runway Incidents → Passenger Injuries
    ('topic_9', 'topic_1'),    # Runway Incidents → Structural Damage
    ('topic_7', 'topic_1'),    # Turbulence → Structural Damage
    ('topic_2', 'topic_7'),    # ATC Communication → Turbulence
    ('topic_4', 'topic_9'),    # Landing Gear → Runway Incidents
    ('topic_13', 'topic_4'),   # Electrical Faults → Landing Gear
    ('topic_6', 'topic_13'),   # Fatigue Crack → Electrical Faults
    ('topic_8', 'topic_13'),   # Fuel System → Electrical Faults
    ('topic_8', 'topic_6'),    # Fuel System → Fatigue Crack
    ('topic_8', 'topic_10'),   # Fuel System → Electrical Wiring
    ('topic_8', 'topic_11'),   # Fuel System → Fan Blade Failure
    ('topic_10', 'topic_16'),  # Electrical Wiring → Smoke in Cabin
    ('topic_10', 'topic_18'),  # Electrical Wiring → Hydraulic Pressure
    ('topic_13', 'topic_16'),  # Electrical Faults → Smoke in Cabin
    ('topic_14', 'topic_15'),  # High EGT → Turbine Damage
    ('topic_14', 'topic_11'),  # High EGT → Fan Blade Failure
    ('topic_14', 'topic_10'),  # High EGT → Electrical Wiring
    ('topic_11', 'topic_15'),  # Fan Blade Failure → Turbine Damage
    ('topic_6', 'topic_11'),   # Fatigue Crack → Fan Blade Failure
]

print(f"DAG: {len(dag_edges)} edges")

# ============================================================================
# 3. BUILD ADJACENCY INFO
# ============================================================================
all_nodes = set()
parents_of = {}   # child → [list of parents]
children_of = {}  # parent → [list of children]

for parent, child in dag_edges:
    all_nodes.add(parent)
    all_nodes.add(child)
    parents_of.setdefault(child, []).append(parent)
    children_of.setdefault(parent, []).append(child)

# Ensure all nodes are tracked (including isolates if any)
for n in all_nodes:
    if n not in parents_of:
        parents_of[n] = []
    if n not in children_of:
        children_of[n] = []

root_nodes = sorted([n for n in all_nodes if len(parents_of[n]) == 0])
child_nodes = sorted([n for n in all_nodes if len(parents_of[n]) > 0])
leaf_nodes = sorted([n for n in all_nodes if len(children_of[n]) == 0])

print(f"Nodes: {len(all_nodes)}")
print(f"Root nodes (no parents):  {len(root_nodes)}")
print(f"Child nodes (have parents): {len(child_nodes)}")
print(f"Leaf nodes (no children): {len(leaf_nodes)}")

# ============================================================================
# 3a. PRINT FULL ADJACENCY STRUCTURE
# ============================================================================
print("\n" + "=" * 80)
print("ADJACENCY STRUCTURE")
print("=" * 80)

# Adjacency matrix
sorted_nodes = sorted(all_nodes, key=lambda x: int(x.split('_')[1]))
n_nodes = len(sorted_nodes)
adj_matrix = pd.DataFrame(0, index=sorted_nodes, columns=sorted_nodes)
for parent, child in dag_edges:
    adj_matrix.loc[parent, child] = 1

print("\nADJACENCY MATRIX (rows=from, cols=to):")
print(adj_matrix.to_string())

print("\n\nNODE DETAILS:")
print(f"{'Node':<12} {'Label':<38} {'Parents':>3} {'Children':>5} {'Type':<8} {'n_obs':>5} {'P(X=1)':>8}")
print("-" * 90)
for node in sorted_nodes:
    n_par = len(parents_of[node])
    n_ch = len(children_of[node])
    if n_par == 0:
        ntype = "ROOT"
    elif n_ch == 0:
        ntype = "LEAF"
    else:
        ntype = "INTER"
    n_obs = int(df[node].sum())
    p = df[node].mean()
    lbl = topic_labels.get(node, '?')
    print(f"{node:<12} {lbl:<38} {n_par:>3} {n_ch:>5} {ntype:<8} {n_obs:>5} {p:>8.4f}")

print("\n\nEDGE LIST WITH LABELS:")
for i, (parent, child) in enumerate(dag_edges, 1):
    p_lbl = topic_labels.get(parent, '?')
    c_lbl = topic_labels.get(child, '?')
    print(f"  {i:>2}. {parent} → {child}  |  {p_lbl} → {c_lbl}")

print("\n\nPARENT STRUCTURE (for CPT computation):")
for node in sorted_nodes:
    pars = parents_of[node]
    if pars:
        cpt_size = 2 ** (len(pars) + 1)
        par_str = ' + '.join(pars)
        print(f"  {node} ← [{par_str}]  →  CPT size: 2^{len(pars)+1} = {cpt_size} entries")
    else:
        print(f"  {node} : ROOT (marginal only)")


# ============================================================================
# 4. MARGINAL PROBABILITIES (ROOT NODES)
# ============================================================================
print("\n" + "=" * 80)
print("MARGINAL PROBABILITIES (Root Nodes)")
print("=" * 80)

marginals = {}
for node in root_nodes:
    p = df[node].mean()
    n = int(df[node].sum())
    marginals[node] = p
    print(f"  P({node}=1) = {p:.6f}  [{topic_labels.get(node,'')}]  (n={n}/{N})")


# ============================================================================
# 5. METHOD 1: MLE WITH LAPLACE SMOOTHING
# ============================================================================
print("\n" + "=" * 80)
print("METHOD 1: MLE with Laplace Smoothing (α=1)")
print("=" * 80)

def compute_cpt_mle(df, child, parents, laplace=1):
    m = len(parents)
    rows = []
    for combo in product([0, 1], repeat=m):
        mask = pd.Series(True, index=df.index)
        for par, val in zip(parents, combo):
            mask &= (df[par] == val)
        count = int(mask.sum())
        count_1 = int((df.loc[mask, child] == 1).sum()) if count > 0 else 0
        p1 = (count_1 + laplace) / (count + 2 * laplace)
        row = {p: v for p, v in zip(parents, combo)}
        row['P(child=1)'] = round(p1, 6)
        row['P(child=0)'] = round(1.0 - p1, 6)
        row['n_obs'] = count
        row['n_child_1'] = count_1
        rows.append(row)
    return pd.DataFrame(rows)

mle_cpts = {}
for node in child_nodes:
    pars = parents_of[node]
    cpt = compute_cpt_mle(df, node, pars)
    mle_cpts[node] = cpt
    print(f"\n  ── {label(node)} ──")
    print(f"  Parents: {[label(p) for p in pars]}")
    print(cpt.to_string(index=False))
    sparse = (cpt['n_obs'] < 5).sum()
    zero = (cpt['n_obs'] == 0).sum()
    if zero > 0:
        print(f"  ⚠ {zero} combinations with ZERO observations (Laplace prior dominates)")
    if sparse > 0:
        print(f"  ⚠ {sparse} combinations with < 5 observations")


# ============================================================================
# 6. METHOD 2: BAYESIAN ESTIMATION (BDeu)
# ============================================================================
print("\n" + "=" * 80)
print("METHOD 2: Bayesian Estimation (BDeu Prior, ESS=5)")
print("=" * 80)

def compute_cpt_bdeu(df, child, parents, ess=5):
    m = len(parents)
    q = 2 ** m
    r = 2
    alpha_prime = ess / (q * r)
    rows = []
    for combo in product([0, 1], repeat=m):
        mask = pd.Series(True, index=df.index)
        for par, val in zip(parents, combo):
            mask &= (df[par] == val)
        count = int(mask.sum())
        count_1 = int((df.loc[mask, child] == 1).sum()) if count > 0 else 0
        p1 = (count_1 + alpha_prime) / (count + 2 * alpha_prime)
        row = {p: v for p, v in zip(parents, combo)}
        row['P(child=1)'] = round(p1, 6)
        row['P(child=0)'] = round(1.0 - p1, 6)
        row['n_obs'] = count
        row['alpha_prime'] = round(alpha_prime, 6)
        rows.append(row)
    return pd.DataFrame(rows)

bdeu_cpts = {}
for node in child_nodes:
    pars = parents_of[node]
    cpt = compute_cpt_bdeu(df, node, pars, ess=5)
    bdeu_cpts[node] = cpt
    ap = cpt['alpha_prime'].iloc[0]
    print(f"\n  ── {label(node)} ──")
    print(f"  Parents: {pars}, α'={ap:.4f}")
    print(cpt.drop(columns=['alpha_prime']).to_string(index=False))


# ============================================================================
# 7. METHOD 3: BETA CDF APPROXIMATION (for nodes with 2+ parents)
# ============================================================================
print("\n" + "=" * 80)
print("METHOD 3: Beta CDF Approximation (Zhang & Mahadevan, 2021)")
print("  NOTE: Only applied to nodes with ≥ 2 parents")
print("=" * 80)

def estimate_individual_cp(df, child, parent):
    mask = df[parent] == 1
    n = mask.sum()
    if n == 0:
        return 0.0
    return (df.loc[mask, child] == 1).sum() / n

def calibrate_beta(ind_cps):
    probs = np.array(list(ind_cps.values()))
    total = probs.sum()
    if total == 0 or len(probs) < 2:
        return 1.0, 2.0, np.inf
    lambdas = probs / total
    targets = probs
    def obj(params):
        a, b = params
        if a <= 0 or b <= 0:
            return 1e10
        return np.sum((beta_dist.cdf(lambdas, a, b) - targets) ** 2)
    best = None
    for a0, b0 in [(1,2),(0.5,1),(2,1),(1,1),(0.5,3),(2,3)]:
        res = minimize(obj, [a0,b0], method='Nelder-Mead', options={'maxiter':20000})
        if best is None or res.fun < best.fun:
            best = res
    return best.x[0], best.x[1], best.fun

def beta_cdf_cpt(child, parents, ind_cps, alpha, beta_p):
    total = sum(ind_cps.values())
    rows = []
    for combo in product([0, 1], repeat=len(parents)):
        active = [p for p, v in zip(parents, combo) if v == 1]
        if len(active) == 0:
            p1 = 0.0
            lam = 0.0
        elif len(active) == len(parents):
            p1 = 1.0
            lam = 1.0
        else:
            s = sum(ind_cps[p] for p in active)
            lam = s / total if total > 0 else 0
            p1 = float(beta_dist.cdf(lam, alpha, beta_p))
        row = {p: v for p, v in zip(parents, combo)}
        row['lambda'] = round(lam, 6)
        row['P(child=1)'] = round(p1, 6)
        row['P(child=0)'] = round(1.0 - p1, 6)
        rows.append(row)
    return pd.DataFrame(rows)

beta_cpts = {}
beta_params = {}

for node in child_nodes:
    pars = parents_of[node]
    if len(pars) < 2:
        print(f"\n  ── {label(node)} ── SKIPPED (only 1 parent, use MLE/BDeu)")
        continue
    
    ind_cps = {p: estimate_individual_cp(df, node, p) for p in pars}
    alpha, beta_p, mse = calibrate_beta(ind_cps)
    beta_params[node] = (alpha, beta_p, mse)
    cpt = beta_cdf_cpt(node, pars, ind_cps, alpha, beta_p)
    beta_cpts[node] = cpt
    
    print(f"\n  ── {label(node)} ──")
    print(f"  Parents: {pars}")
    print(f"  Individual P(ω|e_i):")
    for p, cp in ind_cps.items():
        nc = int(((df[node]==1)&(df[p]==1)).sum())
        np_ = int(df[p].sum())
        print(f"    P({node}|{p}) = {cp:.4f}  ({nc}/{np_})")
    print(f"  Σ P(ω|e_i) = {sum(ind_cps.values()):.4f}")
    print(f"  Calibrated: α={alpha:.5f}, β={beta_p:.5f}, MSE={mse:.2e}")
    print(f"\n  CPT:")
    print(cpt.to_string(index=False))


# ============================================================================
# 8. METHOD COMPARISON (MLE vs BDeu vs Beta CDF)
# ============================================================================
print("\n" + "=" * 80)
print("METHOD COMPARISON")
print("=" * 80)

for node in child_nodes:
    pars = parents_of[node]
    mle = mle_cpts[node]
    bdeu = bdeu_cpts[node]
    has_beta = node in beta_cpts
    
    print(f"\n  ── {label(node)} | parents: {pars} ──")
    header = f"  {'Parents':<40s} | {'MLE':>7s} | {'BDeu':>7s}"
    sep =    f"  {'-'*40} | {'-'*7} | {'-'*7}"
    if has_beta:
        header += f" | {'BetaCDF':>7s} | {'n':>4s}"
        sep += f" | {'-'*7} | {'-'*4}"
    else:
        header += f" | {'n':>4s}"
        sep += f" | {'-'*4}"
    print(header)
    print(sep)
    
    for idx in range(len(mle)):
        combo_str = ', '.join([f"{p}={int(mle.iloc[idx][p])}" for p in pars])
        p_mle = mle.iloc[idx]['P(child=1)']
        p_bdeu = bdeu.iloc[idx]['P(child=1)']
        n_obs = int(mle.iloc[idx]['n_obs'])
        flag = " ⚠" if n_obs < 5 else ""
        
        line = f"  {combo_str:<40s} | {p_mle:>7.4f} | {p_bdeu:>7.4f}"
        if has_beta:
            p_beta = beta_cpts[node].iloc[idx]['P(child=1)']
            line += f" | {p_beta:>7.4f} | {n_obs:>4d}{flag}"
        else:
            line += f" | {n_obs:>4d}{flag}"
        print(line)


# ============================================================================
# 9. RECOMMENDED CPTs (Hybrid: BDeu for ≤3 parents, Beta CDF for 4+)
# ============================================================================
print("\n" + "=" * 80)
print("RECOMMENDED CPTs (Hybrid Strategy)")
print("  • Root nodes: Marginal probability")
print("  • ≤ 3 parents: BDeu Bayesian Estimation (ESS=5)")
print("  • ≥ 4 parents: Beta CDF Approximation")
print("=" * 80)

recommended = {}

for node in sorted_nodes:
    pars = parents_of[node]
    if len(pars) == 0:
        recommended[node] = {'type': 'marginal', 'P(X=1)': marginals[node]}
        print(f"\n  {label(node)}: ROOT")
        print(f"    P({node}=1) = {marginals[node]:.6f}")
    elif len(pars) >= 4 and node in beta_cpts:
        recommended[node] = {'type': 'beta_cdf', 'cpt': beta_cpts[node]}
        print(f"\n  {label(node)}: Beta CDF (4+ parents)")
        print(beta_cpts[node].to_string(index=False))
    else:
        recommended[node] = {'type': 'bdeu', 'cpt': bdeu_cpts[node]}
        print(f"\n  {label(node)}: BDeu")
        print(bdeu_cpts[node].drop(columns=['alpha_prime']).to_string(index=False))


# ============================================================================
# 10. VALIDATION
# ============================================================================
print("\n" + "=" * 80)
print("VALIDATION CHECKS")
print("=" * 80)

# Check 1: DAG acyclicity
print("\n  Check 1: DAG Acyclicity")
from collections import defaultdict, deque
in_degree = defaultdict(int)
graph = defaultdict(list)
for p, c in dag_edges:
    graph[p].append(c)
    in_degree[c] += 1
    if p not in in_degree:
        in_degree[p] = in_degree.get(p, 0)

queue = deque([n for n in all_nodes if in_degree[n] == 0])
topo_order = []
while queue:
    n = queue.popleft()
    topo_order.append(n)
    for ch in graph[n]:
        in_degree[ch] -= 1
        if in_degree[ch] == 0:
            queue.append(ch)

if len(topo_order) == len(all_nodes):
    print(f"    ✓ DAG is acyclic. Topological order: {topo_order}")
else:
    print(f"    ✗ CYCLE DETECTED! Only {len(topo_order)}/{len(all_nodes)} nodes reachable.")

# Check 2: Normalization
print("\n  Check 2: CPT Normalization")
all_ok = True
for node in child_nodes:
    cpt = bdeu_cpts[node]
    sums = cpt['P(child=1)'] + cpt['P(child=0)']
    if np.allclose(sums, 1.0):
        print(f"    ✓ {node}: all rows sum to 1.0")
    else:
        print(f"    ✗ {node}: normalization error!")
        all_ok = False

# Check 3: Sparsity
print("\n  Check 3: Data Sparsity")
for node in child_nodes:
    cpt = mle_cpts[node]
    total = len(cpt)
    zero = (cpt['n_obs'] == 0).sum()
    low = (cpt['n_obs'] < 5).sum()
    pars = parents_of[node]
    method = "Beta CDF recommended" if len(pars) >= 4 else "BDeu adequate"
    print(f"    {node}: {total} combos, {zero} empty, {low} sparse → {method}")

# Check 4: All nodes covered
print(f"\n  Check 4: Coverage")
nodes_in_dag = all_nodes
nodes_in_data = set(df.columns)
missing = nodes_in_dag - nodes_in_data
extra = nodes_in_data - nodes_in_dag
print(f"    Nodes in DAG: {len(nodes_in_dag)}")
print(f"    Nodes in data: {len(nodes_in_data)}")
print(f"    Nodes in DAG but not in data: {missing if missing else 'None'}")
print(f"    Nodes in data but not in DAG: {sorted(extra) if extra else 'None'}")


# ============================================================================
# 11. EXPORT
# ============================================================================
import os
output_dir = r'C:\Users\55279\Desktop\Mestrado\0.Thesis\Codes\Causa_Inference'
os.makedirs(output_dir, exist_ok=True)

excel_path = os.path.join(output_dir, 'CPT_results_final.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    # Adjacency matrix
    adj_matrix.to_excel(writer, sheet_name='Adjacency_Matrix')
    
    # Node info
    node_info = []
    for node in sorted_nodes:
        pars = parents_of[node]
        node_info.append({
            'node': node,
            'label': topic_labels.get(node, ''),
            'n_parents': len(pars),
            'parents': ', '.join(pars) if pars else '',
            'n_children': len(children_of[node]),
            'children': ', '.join(children_of[node]) if children_of[node] else '',
            'type': 'ROOT' if not pars else ('LEAF' if not children_of[node] else 'INTERMEDIATE'),
            'P(X=1)': df[node].mean(),
            'n_occurrences': int(df[node].sum()),
            'recommended_method': 'Marginal' if not pars else ('Beta CDF' if len(pars) >= 4 else 'BDeu (ESS=5)')
        })
    pd.DataFrame(node_info).to_excel(writer, sheet_name='Node_Info', index=False)
    
    # Marginals
    marg_df = pd.DataFrame([
        {'node': n, 'label': topic_labels.get(n,''), 'P(X=1)': p, 'P(X=0)': 1-p, 'n': int(df[n].sum())}
        for n, p in marginals.items()
    ])
    marg_df.to_excel(writer, sheet_name='Root_Marginals', index=False)
    
    # Edge list
    edge_df = pd.DataFrame([
        {'from': p, 'from_label': topic_labels.get(p,''), 'to': c, 'to_label': topic_labels.get(c,'')}
        for p, c in dag_edges
    ])
    edge_df.to_excel(writer, sheet_name='Edge_List', index=False)
    
    # CPTs per node (recommended method)
    for node in child_nodes:
        pars = parents_of[node]
        sheet = f'CPT_{node}'[:31]
        
        if len(pars) >= 4 and node in beta_cpts:
            beta_cpts[node].to_excel(writer, sheet_name=sheet, index=False)
        else:
            bdeu_cpts[node].drop(columns=['alpha_prime']).to_excel(writer, sheet_name=sheet, index=False)
    
    # Beta params summary
    if beta_params:
        bp_df = pd.DataFrame([
            {'node': n, 'label': topic_labels.get(n,''), 'alpha': a, 'beta': b, 'MSE': m}
            for n, (a, b, m) in beta_params.items()
        ])
        bp_df.to_excel(writer, sheet_name='Beta_Parameters', index=False)

print(f"\n  Exported to: {excel_path}")

# Also export adjacency matrix as CSV for easy use
adj_csv = os.path.join(output_dir, 'adjacency_matrix.csv')
adj_matrix.to_csv(adj_csv)
print(f"  Adjacency matrix: {adj_csv}")

print("\n" + "=" * 80)
print("DONE — All CPTs computed and exported.")
print("=" * 80)

Loaded: 596 accidents × 19 topics

DAG: 19 edges
Nodes: 15
Root nodes (no parents):  3
Child nodes (have parents): 12
Leaf nodes (no children): 5

ADJACENCY STRUCTURE

ADJACENCY MATRIX (rows=from, cols=to):
          topic_0  topic_1  topic_2  topic_4  topic_6  topic_7  topic_8  topic_9  topic_10  topic_11  topic_13  topic_14  topic_15  topic_16  topic_18
topic_0         0        0        0        0        0        0        0        0         0         0         0         0         0         0         0
topic_1         0        0        0        0        0        0        0        0         0         0         0         0         0         0         0
topic_2         0        0        0        0        0        1        0        0         0         0         0         0         0         0         0
topic_4         0        0        0        0        0        0        0        1         0         0         0         0         0         0         0
topic_6         0        0        0   